In [1]:
from pathlib import Path
import sys
import pandas as pd

cwd = Path.cwd().resolve()
project_root = cwd if (cwd / "src").is_dir() else cwd.parent

if not (project_root / "src").is_dir():
    raise FileNotFoundError(f"Không tìm thấy thư mục src từ: {cwd}")

project_root_str = str(project_root)
if project_root_str not in sys.path:
    sys.path.insert(0, project_root_str)

print(f"Project root: {project_root}")

Project root: /home/trieu/Intern/SwM_precomputed


In [2]:
from src.data.train_dataset import TrainDataset
from src.data.evaluation_dataset import EvaluationDataset

from torch.utils.data import DataLoader
from src.utils.io import load_pickle
from src.models.sasrec import SASRec
import torch

In [3]:
train_samples = [
    {
        'history': [
            'N8129',
            'N1569',
            'N17686',
        ],
        'target': 'N13008',
    },
    {
        'history': [
            'N63302',
            'N10414',
            'N19347',
            'N31801'
        ],
        'target': 'N55689'
    },
    {
        'history': [
            'N21623',
            'N6233',
            'N14340',
            'N48031',
            'N62285'
        ],
        'target': 'N31739'
    },
    {
        'history': [
            'N31739',
            'N6072',
            'N63045',
            'N23979',
            'N35656',
        ],
        'target': 'N43353'
    },
    
]

In [4]:
padding_id = 0
max_sequence_length = 5
evaluation_samples = []
num_negatives = 5
mapping = load_pickle(project_root / "data/processed/mindsmall_v1/artifacts/news_vector_mapping.pkl")
batch_size = 2
num_blocks = 2
num_heads = 2
dropout = 0.1
embedding_dim = 384
lr = 0.001

In [5]:
train_dataset = TrainDataset(samples=train_samples, max_sequence_length=max_sequence_length, padding_id=padding_id, mapping=mapping, vector_size=384)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

In [6]:
model = SASRec(
    max_sequence_length=max_sequence_length,
    num_blocks=num_blocks,
    num_heads=num_heads,
    dropout=dropout,
    embedding_dim=embedding_dim
)

In [7]:
optimizer = torch.optim.Adam(model.parameters(), lr=lr)

In [8]:
model.train()
total_loss = 0.0
batch_train = next(iter(train_loader))

In [9]:
input_vectors = batch_train['input_vectors']
positive_vectors = batch_train['positive_vectors']
negative_vectors = batch_train['negative_vectors']
optimizer.zero_grad()
outputs = model(
    input_vectors=input_vectors,
    positive_vectors=positive_vectors,
    negative_vectors=negative_vectors,
)
outputs

{'positive_logits': tensor([[ 0.0000,  0.1765,  0.9444,  0.5580,  1.4025],
         [ 0.2451, -0.3148,  0.4426,  1.0694,  1.3575]], grad_fn=<SumBackward1>),
 'negative_logits': tensor([[ 0.0000, -1.9011,  0.8718,  1.0523, -1.5697],
         [ 1.6589, -0.6546,  0.6881,  0.8227, -0.6453]], grad_fn=<SumBackward1>),
 'sequence_output': tensor([[[ 0.0000e+00,  0.0000e+00,  0.0000e+00,  ...,  0.0000e+00,
            0.0000e+00,  0.0000e+00],
          [-2.4085e+00,  3.1478e-01,  1.8811e-01,  ..., -2.0653e-01,
           -1.9822e-01, -6.5935e-01],
          [-6.6371e-01, -1.3494e-01, -1.2546e+00,  ...,  1.2197e+00,
            1.5847e+00, -5.6717e-01],
          [-4.5977e-01, -8.4865e-01,  8.8677e-01,  ...,  1.3644e-01,
            1.9857e-01, -7.1042e-01],
          [-2.1581e+00, -2.9789e-01, -1.9315e-02,  ..., -5.2895e-01,
            8.8785e-01, -6.0516e-01]],
 
         [[ 1.4685e+00, -2.5619e-01, -9.8257e-01,  ...,  2.2614e-01,
            1.1307e+00,  2.0416e-01],
          [-2.3924e+00

In [10]:
input_vectors.shape, positive_vectors.shape, negative_vectors.shape

(torch.Size([2, 5, 384]), torch.Size([2, 5, 384]), torch.Size([2, 5, 384]))

In [11]:
positive_vectors.shape[:-1]

torch.Size([2, 5])

In [12]:
outputs['sequence_output'].shape

torch.Size([2, 5, 384])

In [13]:
positive_logits = outputs['positive_logits']
negative_logits = outputs['negative_logits']

In [14]:
positive_labels = torch.ones_like(positive_logits)
negative_labels = torch.zeros_like(negative_logits)
positive_labels, negative_labels

(tensor([[1., 1., 1., 1., 1.],
         [1., 1., 1., 1., 1.]]),
 tensor([[0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0.]]))

In [15]:
loss_fn = torch.nn.BCEWithLogitsLoss()

In [16]:
positive_loss = loss_fn(positive_logits, positive_labels)
negative_loss = loss_fn(negative_logits, negative_labels)
positive_loss, negative_loss

(tensor(0.4764, grad_fn=<BinaryCrossEntropyWithLogitsBackward0>),
 tensor(0.8550, grad_fn=<BinaryCrossEntropyWithLogitsBackward0>))

In [17]:
if positive_logits.shape != negative_logits.shape:
    raise ValueError("positive_logits and negative_logits must have equal shape")
if positive_vectors.ndim != positive_logits.ndim + 1:
    raise ValueError("positive_vectors must have shape [B, L, T]")
if positive_vectors.shape[:-1] != positive_logits.shape:
    raise ValueError("Token batch/sequence dimensions must match logits")

In [18]:
valid_mask = positive_vectors.ne(0).any(dim=-1).to(positive_loss.dtype)
valid_mask

tensor([[0., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1.]])

In [19]:

loss = (positive_loss + negative_loss) * valid_mask
loss = loss.sum() / valid_mask.sum().clamp_min(1.0)

In [20]:
loss

tensor(1.3314, grad_fn=<DivBackward0>)

In [21]:
val_samples = [
    {
        'history': ['N41241', 'N52026', 'N36360', 'N31550'],
        'target': 'N26125'
    },
    {
        'history': ['N59027'], 
        'target': 'N54803'
    },
    {
        'history': ['N39556','N459','N57300','N4643','N16545','N4501','N10059','N719','N51483','N56446','N32868','N19620','N61084','N10865','N16625','N19638','N31193','N25577','N12096','N48216','N20110','N18275','N28115','N25525','N29453','N16715','N24298','N18355','N4985','N6506','N14761','N1864','N8148','N46811'],
        'target': 'N5981'
    },
    {
        'history': ['N39556','N459','N57300','N4643','N16545','N4501','N10059','N719','N51483','N56446','N32868','N19620','N61084','N10865','N16625','N19638','N31193','N25577','N12096','N48216','N20110','N18275','N28115','N25525','N29453','N16715','N24298','N18355','N4985','N6506','N14761','N1864','N8148','N46811'],
        'target': 'N16120'
    },
]

In [22]:
val_dataset = EvaluationDataset(
    samples=val_samples,
    num_negatives=num_negatives,
    max_sequence_length=max_sequence_length,
    padding_id=padding_id,
    mapping=mapping,
    vector_size=384
)
val_dataloader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False
)
batch_val = next(iter(val_dataloader))
batch_val


{'input_vectors': tensor([[[ 0.0000,  0.0000,  0.0000,  ...,  0.0000,  0.0000,  0.0000],
          [ 0.0802,  0.0029,  0.0789,  ...,  0.0033, -0.1102,  0.1254],
          [ 0.0599,  0.0126,  0.0083,  ..., -0.0621,  0.0019,  0.0680],
          [ 0.0089,  0.0330,  0.0067,  ..., -0.0178, -0.0043, -0.0072],
          [-0.0457, -0.0656, -0.0469,  ...,  0.0916,  0.0327, -0.0145]],
 
         [[ 0.0000,  0.0000,  0.0000,  ...,  0.0000,  0.0000,  0.0000],
          [ 0.0000,  0.0000,  0.0000,  ...,  0.0000,  0.0000,  0.0000],
          [ 0.0000,  0.0000,  0.0000,  ...,  0.0000,  0.0000,  0.0000],
          [ 0.0000,  0.0000,  0.0000,  ...,  0.0000,  0.0000,  0.0000],
          [-0.0120,  0.0106, -0.0459,  ..., -0.0938, -0.0545,  0.0375]]]),
 'candidate_vectors': tensor([[[ 0.0004,  0.0537,  0.1062,  ..., -0.0984, -0.1167,  0.0801],
          [ 0.0124, -0.1000,  0.0053,  ..., -0.0172, -0.0179,  0.0149],
          [-0.1157,  0.1164,  0.0808,  ..., -0.0259, -0.1376,  0.0367],
          [ 0.0159, 

In [23]:
total_hits = 0.0
total_ndcg = 0.0
total_samples = 0

In [24]:
input_vectors = batch_val["input_vectors"]
candidate_vectors = batch_val["candidate_vectors"]

In [25]:
sequence_output, _ = model._run_blocks(input_vectors)

In [26]:
from src.models.sasrec import sequence_padding_mask

In [27]:
padding_mask = sequence_padding_mask(input_vectors)
positions = torch.arange(input_vectors.size(1))
positions, padding_mask

(tensor([0, 1, 2, 3, 4]),
 tensor([[False,  True,  True,  True,  True],
         [False, False, False, False,  True]]))

In [28]:
positions.unsqueeze(0).shape, padding_mask.shape

(torch.Size([1, 5]), torch.Size([2, 5]))

In [29]:
positions.unsqueeze(0).masked_fill(~padding_mask, -1)

tensor([[-1,  1,  2,  3,  4],
        [-1, -1, -1, -1,  4]])

In [30]:
positions.unsqueeze(0).masked_fill(~padding_mask, -1).max(dim=1)

torch.return_types.max(
values=tensor([4, 4]),
indices=tensor([4, 4]))

In [31]:
last_indices = positions.unsqueeze(0).masked_fill(~padding_mask, -1).max(dim=1).values
last_indices

tensor([4, 4])

In [32]:
type(sequence_output), sequence_output.shape, last_indices.shape

(torch.Tensor, torch.Size([2, 5, 384]), torch.Size([2]))

In [33]:
torch.arange(input_vectors.size(0)), last_indices

(tensor([0, 1]), tensor([4, 4]))

In [34]:
sequence_output[0,0,0]

tensor(0., grad_fn=<SelectBackward0>)

In [35]:
sequence_output[
    [0, 1], [4, 4]
]

tensor([[-6.9278e-01, -4.2237e-01, -1.4661e-01,  1.5653e-01,  4.5188e-01,
         -1.4476e-01,  1.0135e+00, -1.9998e+00, -7.7191e-01,  7.2833e-01,
          2.1872e+00,  1.5461e+00, -2.1057e+00, -4.8481e-01, -6.1364e-01,
         -1.1602e+00, -3.2601e-01, -6.1169e-02,  7.7055e-02, -1.6004e-01,
         -8.3448e-01,  1.4956e+00, -1.7536e+00, -7.6357e-02, -2.5500e-01,
          2.1052e-01,  1.1001e-01,  1.6891e+00, -3.1079e-01,  2.3396e+00,
          1.1391e+00,  9.5442e-01, -6.1870e-01, -7.7004e-01, -1.5801e-01,
         -6.8556e-01,  1.7935e-01,  8.9028e-01, -1.2210e+00, -4.4922e-01,
          1.5663e+00,  1.1559e+00, -5.9266e-01,  2.2740e+00,  2.9329e-01,
         -1.3252e+00,  1.7915e+00, -3.0348e-01,  1.4697e-01,  3.5659e-01,
         -1.4191e-01,  8.0977e-02,  5.6857e-01, -9.9968e-01,  7.0638e-01,
          5.5921e-01, -3.3943e-01, -1.8582e+00, -2.0703e+00,  8.4829e-01,
         -9.0166e-01, -5.4382e-01, -1.0761e+00, -2.1472e+00, -1.1565e+00,
         -1.7079e+00,  9.9885e-01,  4.

In [36]:
last_hidden = sequence_output[
    torch.arange(
        input_vectors.size(0),
    ),
    last_indices
]
last_hidden.shape, last_hidden

(torch.Size([2, 384]),
 tensor([[-6.9278e-01, -4.2237e-01, -1.4661e-01,  1.5653e-01,  4.5188e-01,
          -1.4476e-01,  1.0135e+00, -1.9998e+00, -7.7191e-01,  7.2833e-01,
           2.1872e+00,  1.5461e+00, -2.1057e+00, -4.8481e-01, -6.1364e-01,
          -1.1602e+00, -3.2601e-01, -6.1169e-02,  7.7055e-02, -1.6004e-01,
          -8.3448e-01,  1.4956e+00, -1.7536e+00, -7.6357e-02, -2.5500e-01,
           2.1052e-01,  1.1001e-01,  1.6891e+00, -3.1079e-01,  2.3396e+00,
           1.1391e+00,  9.5442e-01, -6.1870e-01, -7.7004e-01, -1.5801e-01,
          -6.8556e-01,  1.7935e-01,  8.9028e-01, -1.2210e+00, -4.4922e-01,
           1.5663e+00,  1.1559e+00, -5.9266e-01,  2.2740e+00,  2.9329e-01,
          -1.3252e+00,  1.7915e+00, -3.0348e-01,  1.4697e-01,  3.5659e-01,
          -1.4191e-01,  8.0977e-02,  5.6857e-01, -9.9968e-01,  7.0638e-01,
           5.5921e-01, -3.3943e-01, -1.8582e+00, -2.0703e+00,  8.4829e-01,
          -9.0166e-01, -5.4382e-01, -1.0761e+00, -2.1472e+00, -1.1565e+00,
  

In [37]:
candidate_vectors.shape, last_hidden.unsqueeze(1).shape

(torch.Size([2, 6, 384]), torch.Size([2, 1, 384]))

In [38]:
scores = (candidate_vectors * last_hidden.unsqueeze(1)).sum(dim=-1)

In [39]:
scores

tensor([[ 1.1559, -0.6935, -0.7352, -0.6238, -0.9244,  0.1347],
        [ 0.3998, -1.0273, -0.1797, -0.7203, -1.0309, -0.5438]],
       grad_fn=<SumBackward1>)

In [42]:
positive_scores = scores[:, 0].unsqueeze(1)
positive_scores.shape, positive_scores

(torch.Size([2, 1]),
 tensor([[1.1559],
         [0.3998]], grad_fn=<UnsqueezeBackward0>))

In [49]:
ranks = (scores > positive_scores).sum(dim=1) + 1
ranks

tensor([1, 1])

In [51]:
hits = (
        ranks <= 10
    ).float()
hits

tensor([1., 1.])

In [53]:
ndcg = torch.where(
        ranks <= 10,
        1.0 / torch.log2(
            ranks.float() + 1.0
        ),
        torch.zeros_like(
            ranks,
            dtype=torch.float,
        ),
    )
ndcg

tensor([1., 1.])